### Nandu – Lösung Schritt für Schritt

*42. Bundeswettbewerb Informatik (2023), 1. Runde, Aufgabe 4*

Nora hat eine Kiste mit elektronischen Spielzeugbausteinen: weiße, blaue und rote. Die weißen und blauen Bausteine haben auf
einer Seite einen oberen und einen unteren Lichtsensor. Die roten Bausteine haben nur oben einen Lichtsensor, können aber gedreht
werden. Alle Bausteine haben auf der gegenüberliegenden Seite eine obere und eine untere LED.

- **Weiß:** Die LEDs leuchten immer beide, außer wenn beide Sensoren bestrahlt werden; dann sind beide LEDs aus.
- **Rot:** Beide LEDs leuchten, wenn kein Licht auf den Sensor fällt, sonst nicht.
- **Blau:** Das Licht wird einfach weitergegeben: Die obere LED leuchtet genau dann, wenn der obere Sensor bestrahlt wird, die
  untere genau dann, wenn der untere Sensor bestrahlt wird.

Nora steckt die Bausteine aneinander. Das Bild auf dem Aufgabenblatt zeigt die Funktion der Bausteine und zwei Konstruktionen mit
den Lichtquellen Q1, Q2 und den LEDs L1, L2.

#### Aufgabe 4

Schreibe ein Programm, das eine Beschreibung einer Konstruktion einliest und angibt, wie die LEDs der ganz rechts (in den Dateien:
ganz unten) liegenden Bausteine auf die Zustände der Lichtquellen reagieren. Wende das Programm auf alle Beispiele von den
BWINF-Webseiten an und dokumentiere die Ergebnisse. (Die Zusatzaufgabe, eine interaktive Umgebung im Browser, lassen wir hier weg.)

#### Beispieldaten (`beispieldaten/nandu1.txt` bis `nandu5.txt`)

- 1. Zeile: Breite `n` und Höhe `m` der Konstruktion
- folgende `m` Zeilen: die Konstruktion; die erste Zeile enthält die Lichtquellen `Q1`, `Q2`, ..., die letzte die LEDs `L1`, `L2`, ...
- jeder Baustein ist zwei Felder breit: weiß `W W`, blau `B B`, rot `R r` oder `r R`
  (großes `R` = Seite mit dem Lichtsensor, kleines `r` = ohne Sensor); `X` = leeres Feld
- In den Dateien läuft das Licht von **oben nach unten** (im Bild von links nach rechts).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation Reihe für Reihe:** Das Verhalten eines Bausteins hängt nur von den Lichtern der Reihe direkt darüber ab. Also
  berechnen wir Reihe für Reihe, in welchen Spalten Licht nach unten abgegeben wird.
- **Wahrheitstabelle durch Ausprobieren:** Bei `k` Lichtquellen gibt es $2^k$ Zustände (jede Quelle ist an oder aus). Wir
  probieren alle durch. Bei den Beispielen sind es höchstens 6 Quellen, also höchstens 64 Zustände.
- **Mengen:** Wir merken uns nur die Menge der Spalten, in denen Licht ankommt. Ob eine Spalte darin vorkommt, fragt man mit `in`.
- **Logik:** Weiß ist ein NAND-Gatter, Rot ein NOT-Gatter, Blau reicht zwei getrennte Signale durch.

**Nützliche Funktionen und Bibliotheken**
- `itertools.product([0, 1], repeat=k)`: alle Kombinationen aus `k` Nullen und Einsen
- `set` (Menge) mit `add` und `in`
- `split()`: Zeile in Wörter zerlegen (`X`, `W`, `R`, `Q1` usw. sind durch Leerzeichen getrennt)

## Die Idee

1. **Einlesen:** Jede Zeile in Wörter zerlegen. Ein Wort entspricht einem Feld.
2. **Bausteine erkennen:** Eine Zeile von links nach rechts durchgehen. Trifft man auf einen Baustein (kein `X`), beginnt dort
   ein neuer Baustein, und das nächste Feld gehört noch dazu: das überspringen wir. Nur so lassen sich zwei weiße Bausteine
   nebeneinander (`W W W W`) richtig trennen.
3. **Eine Reihe simulieren:** Gegeben die Menge der Spalten, in denen oben Licht ankommt, berechne die Menge der Spalten, in denen
   die Bausteine Licht nach unten abgeben.
4. **Alle Zustände der Lichtquellen** durchprobieren, von oben nach unten simulieren und an den `L`-Feldern ablesen.

## Schritt 1: Einlesen

Wir lesen die Datei und zerlegen jede Zeile der Konstruktion mit `split()` in Wörter. `konstruktion` ist eine Liste von
Zeilen, jede Zeile eine Liste von Wörtern. (In `nandu5.txt` hängen Leerzeichen am Zeilenende, `split()` stört das nicht.)

In [ ]:
def lies_konstruktion(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    n, m = [int(wort) for wort in zeilen[0].split()]
    konstruktion = []
    for zeile in zeilen[1:1 + m]:
        konstruktion.append(zeile.split())
    return n, m, konstruktion


n, m, konstruktion = lies_konstruktion('beispieldaten/nandu1.txt')
print(n, 'Spalten,', m, 'Zeilen')
for zeile in konstruktion:
    print(zeile)

4 Spalten, 6 Zeilen
['X', 'Q1', 'Q2', 'X']
['X', 'W', 'W', 'X']
['r', 'R', 'R', 'r']
['X', 'B', 'B', 'X']
['X', 'W', 'W', 'X']
['X', 'L1', 'L2', 'X']


## Schritt 2: Bausteine einer Zeile erkennen

Wir gehen die Wörter einer Zeile von links nach rechts durch (`s` ist die Spalte). Ist das Wort `X`, geht es eine Spalte
weiter. Sonst beginnt hier ein Baustein, der die Spalten `s` und `s + 1` belegt; danach machen wir bei `s + 2` weiter.
Wir speichern jeden Baustein als Tupel `(art, s)` mit `art` gleich `'W'`, `'B'`, `'RL'` oder `'RR'`:

- `'W'` weiß, `'B'` blau (beide Wörter eines Bausteins sind gleich).
- Rot: Das **große `R`** markiert den Sensor. `R r` heißt, der Sensor ist **links** (`'RL'`), `r R` heißt, er ist **rechts**
  (`'RR'`). Das erste Wort des Bausteins reicht zur Unterscheidung.

In [ ]:
def bausteine_der_zeile(wortliste):
    bausteine = []
    s = 0
    while s < len(wortliste):
        wort = wortliste[s]
        if wort == 'X':
            s = s + 1
        else:
            if wort == 'R':
                art = 'RL'
            elif wort == 'r':
                art = 'RR'
            else:
                art = wort                  # 'W' oder 'B'
            bausteine.append((art, s))
            s = s + 2                       # der Baustein belegt zwei Felder
    return bausteine


for zeile in konstruktion[1:-1]:
    print(bausteine_der_zeile(zeile))

[('W', 1)]
[('RR', 0), ('RL', 2)]
[('B', 1)]
[('W', 1)]


## Schritt 3: Eine Reihe simulieren

`licht_oben` ist die Menge der Spalten, in denen von oben Licht auf die Reihe fällt (von der Reihe darüber oder von den
Lichtquellen). Ein Baustein bei Spalte `s` hat seine Sensoren in den Spalten `s` und `s + 1`, und dort sind auch seine LEDs. Er gibt
nach unten Licht an diesen Spalten ab:

| Baustein | Licht unten bei |
|---|---|
| Weiß | `s` und `s + 1`, außer wenn **beide** Sensoren Licht haben |
| Blau | `s`, wenn `s` Licht hat; `s + 1`, wenn `s + 1` Licht hat |
| Rot, Sensor links | `s` und `s + 1`, wenn bei `s` **kein** Licht ist |
| Rot, Sensor rechts | `s` und `s + 1`, wenn bei `s + 1` **kein** Licht ist |

Die Funktion gibt die neue Menge zurück. Licht wird nur dort weitergegeben, wo unten direkt ein Sensor sitzt: Felder ohne Baustein
(`X`) tragen nichts bei, und Licht, das auf ein `X` fällt, geht verloren.

In [ ]:
def simuliere_zeile(bausteine, licht_oben):
    licht_unten = set()
    for art, s in bausteine:
        links = s in licht_oben
        rechts = (s + 1) in licht_oben
        if art == 'W':
            if not (links and rechts):
                licht_unten.add(s)
                licht_unten.add(s + 1)
        elif art == 'B':
            if links:
                licht_unten.add(s)
            if rechts:
                licht_unten.add(s + 1)
        elif art == 'RL':
            if not links:
                licht_unten.add(s)
                licht_unten.add(s + 1)
        else:                               # 'RR'
            if not rechts:
                licht_unten.add(s)
                licht_unten.add(s + 1)
    return licht_unten


# Ein weißer Baustein in den Spalten 1 und 2 (wie in nandu1)
weiss = [('W', 1)]
print(simuliere_zeile(weiss, {1, 2}))      # beide Sensoren beleuchtet: nichts
print(simuliere_zeile(weiss, {1}))         # nur einer: beide LEDs an
print(simuliere_zeile(weiss, set()))       # keiner: beide LEDs an

set()
{1, 2}
{1, 2}


## Schritt 4: Eine Konstruktion für einen Zustand der Quellen auswerten

Die erste Zeile der Konstruktion enthält die Lichtquellen `Q1`, `Q2`, ... und die letzte die LEDs `L1`, `L2`, ... . Die Nummer
steht im Namen; wir ordnen deshalb **nach der Nummer** (nicht nach der Position in der Zeile). Das tut `spalten_der_namen`.
Für eine Eingabe (Liste von 0 und 1, die `i`-te Zahl gehört zur Quelle `Q(i+1)`) gehen wir so vor:

1. `licht` ist die Menge der Spalten der Quellen, die an sind.
2. Für jede Zeile dazwischen: `licht = simuliere_zeile(bausteine_der_zeile(zeile), licht)`.
3. Nach der letzten Baustein-Zeile liest man an den Spalten der `L`-Felder ab, ob dort Licht ankommt.

In [ ]:
def spalten_der_namen(zeile, buchstabe):
    """Spalten der Felder, die mit buchstabe beginnen (Q oder L), sortiert nach der Nummer im Namen."""
    paare = []
    for s in range(len(zeile)):
        if zeile[s][0] == buchstabe:
            paare.append((int(zeile[s][1:]), s))
    paare.sort()
    return [s for nummer, s in paare]


def werte_aus(konstruktion, eingabe):
    quellen = spalten_der_namen(konstruktion[0], 'Q')
    leds = spalten_der_namen(konstruktion[-1], 'L')
    licht = set()
    for i in range(len(quellen)):
        if eingabe[i] == 1:
            licht.add(quellen[i])
    for zeile in konstruktion[1:-1]:
        licht = simuliere_zeile(bausteine_der_zeile(zeile), licht)
    ausgabe = []
    for s in leds:
        if s in licht:
            ausgabe.append(1)
        else:
            ausgabe.append(0)
    return ausgabe


print(werte_aus(konstruktion, [0, 0]))
print(werte_aus(konstruktion, [1, 1]))

[1, 1]
[0, 0]


## Schritt 5: Alle Zustände der Quellen und die Tabelle

`itertools.product([0, 1], repeat=k)` liefert alle $2^k$ Eingaben in der Reihenfolge einer Wahrheitstabelle (`Q1` ändert sich am
langsamsten). Für jede Eingabe berechnen wir die Ausgabe und drucken eine Zeile `Q1 ... Qk | L1 ... Ll` (1 = An, 0 = Aus).

In [ ]:
from itertools import product


def tabelle(konstruktion):
    """Gibt alle Zeilen (Eingabe, Ausgabe) der Wahrheitstabelle zurück."""
    k = len(spalten_der_namen(konstruktion[0], 'Q'))
    zeilen = []
    for eingabe in product([0, 1], repeat=k):
        zeilen.append((list(eingabe), werte_aus(konstruktion, eingabe)))
    return zeilen


def drucke_tabelle(konstruktion, maximal=20):
    k = len(spalten_der_namen(konstruktion[0], 'Q'))
    anzahl_leds = len(spalten_der_namen(konstruktion[-1], 'L'))
    kopf = ['Q' + str(i + 1) for i in range(k)] + ['|'] + ['L' + str(i + 1) for i in range(anzahl_leds)]
    print(*kopf)
    zeilen = tabelle(konstruktion)
    for eingabe, ausgabe in zeilen[:maximal]:
        print(*(eingabe + ['|'] + ausgabe))
    if len(zeilen) > maximal:
        print('...', len(zeilen), 'Zeilen insgesamt')


drucke_tabelle(konstruktion)

Q1 Q2 | L1 L2
0 0 | 1 1
0 1 | 1 1
1 0 | 1 1
1 1 | 0 0


Das ist die Tabelle zu `nandu1.txt`: Nur wenn **beide** Quellen leuchten, bleiben `L1` und `L2` aus.

Zum Vergleich die Konstruktion links auf dem Aufgabenblatt, zwei weiße Bausteine hintereinander. Sie verhält sich wie ein UND: Nur bei
An/An leuchten beide LEDs, wie in der Tabelle der Aufgabe.

In [ ]:
und_schaltung = [['X', 'Q1', 'Q2', 'X'],
                 ['X', 'W', 'W', 'X'],
                 ['X', 'W', 'W', 'X'],
                 ['X', 'L1', 'L2', 'X']]
drucke_tabelle(und_schaltung)

Q1 Q2 | L1 L2
0 0 | 0 0
0 1 | 0 0
1 0 | 0 0
1 1 | 1 1


## Alle Beispiele

Für jede Beispieldatei die Wahrheitstabelle. Bei `nandu5` hat die Tabelle 64 Zeilen, wir drucken nur die ersten 12. Alle Zeilen aller fünf Tabellen wurden
mit den Tabellen der offiziellen Lösungshinweise verglichen und stimmen überein.

In [ ]:
for nummer in range(1, 6):
    n, m, konstruktion = lies_konstruktion('beispieldaten/nandu' + str(nummer) + '.txt')
    print('nandu' + str(nummer) + '.txt  (' + str(n) + ' x ' + str(m) + ')')
    drucke_tabelle(konstruktion, 12 if nummer == 5 else 20)
    print()

nandu1.txt  (4 x 6)
Q1 Q2 | L1 L2
0 0 | 1 1
0 1 | 1 1
1 0 | 1 1
1 1 | 0 0

nandu2.txt  (6 x 8)
Q1 Q2 | L1 L2
0 0 | 0 1
0 1 | 0 1
1 0 | 0 1
1 1 | 1 0

nandu3.txt  (8 x 7)
Q1 Q2 Q3 | L1 L2 L3 L4
0 0 0 | 1 0 0 1
0 0 1 | 1 0 0 0
0 1 0 | 1 0 1 1
0 1 1 | 1 0 1 0
1 0 0 | 0 1 0 1
1 0 1 | 0 1 0 0
1 1 0 | 0 1 1 1
1 1 1 | 0 1 1 0

nandu4.txt  (8 x 8)
Q1 Q2 Q3 Q4 | L1 L2
0 0 0 0 | 0 0
0 0 0 1 | 0 0
0 0 1 0 | 0 1
0 0 1 1 | 0 0
0 1 0 0 | 1 0
0 1 0 1 | 1 0
0 1 1 0 | 1 1
0 1 1 1 | 1 0
1 0 0 0 | 0 0
1 0 0 1 | 0 0
1 0 1 0 | 0 1
1 0 1 1 | 0 0
1 1 0 0 | 0 0
1 1 0 1 | 0 0
1 1 1 0 | 0 1
1 1 1 1 | 0 0

nandu5.txt  (22 x 14)
Q1 Q2 Q3 Q4 Q5 Q6 | L1 L2 L3 L4 L5
0 0 0 0 0 0 | 0 0 0 1 0
0 0 0 0 0 1 | 0 0 0 1 0
0 0 0 0 1 0 | 0 0 0 1 1
0 0 0 0 1 1 | 0 0 0 1 1
0 0 0 1 0 0 | 0 0 1 0 0
0 0 0 1 0 1 | 0 0 1 0 0
0 0 0 1 1 0 | 0 0 0 1 1
0 0 0 1 1 1 | 0 0 0 1 1
0 0 1 0 0 0 | 0 0 0 1 0
0 0 1 0 0 1 | 0 0 0 1 0
0 0 1 0 1 0 | 0 0 0 1 1
0 0 1 0 1 1 | 0 0 0 1 1
... 64 Zeilen insgesamt



### Was die Tabellen zeigen

- `nandu1`: NAND. Nur bei An/An sind beide LEDs aus.
- `nandu2`: `L1` leuchtet genau dann, wenn `Q1` und `Q2` an sind (UND), `L2` ist das Gegenteil davon (NAND).
- `nandu3`: `L1` ist das Gegenteil von `Q1`, `L2` folgt `Q1`, `L3` folgt `Q2`, `L4` ist das Gegenteil von `Q3`.
- `nandu4`: `L1` leuchtet genau dann, wenn `Q2` an und `Q1` aus ist; `L2` genau dann, wenn `Q3` an und `Q4` aus ist.
- `nandu5`: Die größte Konstruktion (22 x 14, 6 Quellen, 5 LEDs); 64 Zustände mit je 12 Baustein-Reihen sind schnell simuliert.
  Die gezeigten ersten Zeilen sind unauffällig; die vollständige Tabelle stimmt mit den Hinweisen überein.

## Der Pseudocode der Lösungshinweise und weiße Bausteine

Der Pseudocode in den Lösungshinweisen (Algorithmus 6) schreibt für weiße Bausteine „Licht, wenn **einer von beiden** Sensoren
Licht hat“. Das widerspricht der Aufgabe (weiß leuchtet **immer**, außer wenn **beide** Sensoren bestrahlt werden) und auch den Tabellen
der Hinweise selbst: In der Tabelle zu `nandu1` leuchten bei 0 0 beide LEDs. Wir halten uns an den Aufgabentext; unsere Tabellen
stimmen mit denen der Hinweise überein.

**Aufwand:** Pro Zustand hat jede Reihe höchstens $n/2$ Bausteine, also $O(n \cdot m)$; für alle Zustände $O(2^k \cdot n \cdot m)$.

**MiniZinc:** Für diese Aufgabe lassen wir es weg. Es gibt nichts zu optimieren und nichts zu suchen: Die Ausgaben ergeben sich
eindeutig durch Simulation von oben nach unten, und mehr als 64 Zustände sind nie zu prüfen.